In [1]:
from pathlib import Path

ROOT = Path(".").resolve().parents[1]
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
from rich import print as rprint

In [3]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [4]:
from src.application.contracts import PipelineRequest
from src.application import ViRAGEPipeline
from src.application.settings import ViRAGESettings

tmp_path = Path("../demo_data/tmp_folder").resolve()
data_path = Path("../demo_data/Iris.csv").resolve()
settings = ViRAGESettings(artifact_root=tmp_path / "artifacts")
pipeline = ViRAGEPipeline(settings)
query="Show the sales trend over time"
request = PipelineRequest(query=query, data_path=data_path.as_posix())
result = pipeline.invoke(request)

In [5]:
rprint(result)

PipelineResult(
    run_id='29ba261e1d6f489cbfe567b5ab86bd51',
    query='Show the sales trend over time',
    data_path='D:/programming/projects/mas_rag/demo_data/Iris.csv',
    case_type=<ChartCaseType.CANONICAL: 'canonical'>,
    query_understanding=QueryUnderstandingResult(
        intent='Show the sales trend over time',
        requested_operations=['trend analysis'],
        candidate_charts=['line'],
        constraints=[],
        case_type=<ChartCaseType.CANONICAL: 'canonical'>,
        confidence=0.8
    ),
    planning=PlanningResult(
        mode=<ChartCaseType.CANONICAL: 'canonical'>,
        steps=[
            PlanningStep(
                name='profile_the_dataset_and_confirm_field',
                description='Profile the dataset and confirm field types relevant to the request.'
            ),
            PlanningStep(
                name='prepare_a_cleaned_analysis_ready_version',
                description='Prepare a cleaned analysis-ready version of the data.'
            ),
            PlanningStep(
                name='retrieve_concise_charting_guidance_for_the',
                description='Retrieve concise charting guidance for the selected chart family.'
            ),
            PlanningStep(
                name='build_the_primary_requested_chart_using',
                description='Build the primary requested chart using the leading chart family: line.'
            ),
            PlanningStep(
                name='execute_plotting_code_and_collect_numeric',
                description='Execute plotting code and collect numeric summaries from the run.'
            ),
            PlanningStep(
                name='read_chart_structure,_extract_facts_and',
                description='Read chart structure, extract facts and verify that final statements are 
evidence-backed.'
            )
        ],
        success_criteria=[
            'At least one valid canonical chart is produced, preferably among: line.',
            'Generated charts are readable and consistent with the request.',
            'Final statements reference execution metrics or chart evidence.'
        ]
    ),
    data_profile=DataProfile(
        row_count=150,
        col_count=6,
        columns=[
            DataColumnProfile(name='Id', dtype='numeric', missing_ratio=0.0, unique_count=150),
            DataColumnProfile(name='SepalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=35),
            DataColumnProfile(name='SepalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=23),
            DataColumnProfile(name='PetalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=43),
            DataColumnProfile(name='PetalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=22),
            DataColumnProfile(name='Species', dtype='categorical', missing_ratio=0.0, unique_count=3)
        ],
        likely_numeric_columns=['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm'],
        likely_categorical_columns=['Species'],
        likely_time_columns=[],
        quality_notes=["Column 'Id' looks like an identifier."]
    ),
    data_preparation=DataPreparationResult(
        output_path='D:/programming/projects/mas_rag/demo_data/tmp_folder/artifacts/29ba261e1d6f489cbfe567b5ab86bd5
1/cleaned_data.csv',
        operations=[],
        row_count=150,
        col_count=6
    ),
    visrag=VisRAGResult(
        recommendations=[
            VisRAGRecommendation(
                chart_family='line',
                rationale='Selected as a conservative fallback based on the request and available data shape.',
                priority=1,
                score=1.45,
                supporting_example_ids=[],
                supporting_corpora=[]
            )
        ],
        rules=[
            'Use clear titles and axis labels.',
            'Avoid overcrowded visuals.',
            'Prefer readable defaults.',
            'Prefer chart families supported by both the data profile an

In [6]:
from langchain_ollama import ChatOllama

LLM_MODEL = "gemma3:1b"  # или "llama3.2:1b"
# LLM_MODEL = "llama3.2:1b"       # или "llama3.2:1b"
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,
)

In [7]:
from src.infrastructure import RuntimeContext

runtime = RuntimeContext(settings=settings, llm=llm)

# QueryUnderstandingService

In [10]:
from src.services import QueryUnderstandingService

qu = QueryUnderstandingService().invoke(runtime=runtime, user_context=request.user_context, query=request.query)

In [12]:
rprint("query:", request.query)
rprint(qu)

query: Show the sales trend over time

QueryUnderstandingResult(
    intent='Trend Analysis',
    requested_operations=['Time Series Visualization'],
    candidate_charts=['Line Chart', 'Bar Chart'],
    constraints=['Time Period: Over time'],
    case_type=<ChartCaseType.NON_CANONICAL: 'non_canonical'>,
    confidence=0.95
)

# CanonicalPlanningService

In [13]:
from src.services import CanonicalPlanningService

cp = CanonicalPlanningService().invoke(runtime=runtime, query_understanding=qu)

In [14]:
rprint(cp)

PlanningResult(
    mode=<ChartCaseType.CANONICAL: 'canonical'>,
    steps=[
        PlanningStep(
            name='1._**define_time_period:**_select_a',
            description='1. **Define Time Period:** Select a specific time period (e.g., last 30 days).'
        ),
        PlanningStep(
            name='2._**data_preparation:**_ensure_data_is',
            description='2. **Data Preparation:** Ensure data is clean and appropriately formatted for 
visualization (e.g., remove outliers, handle missing values if necessary).'
        ),
        PlanningStep(
            name='3._**line_chart:**_create_a_line',
            description='3. **Line Chart:** Create a Line Chart with the chosen time period as the x-axis and the 
data values as the y-axis.'
        ),
        PlanningStep(
            name='4._**verify_data:**_confirm_the_chart',
            description='4. **Verify Data:** Confirm the chart accurately represents the data and the chosen time 
period.'
        ),
        PlanningStep(
            name='5._**check_for_trends:**_examine_the',
            description='5. **Check for Trends:** Examine the line chart for any obvious trends (increasing, 
decreasing, cyclical).'
        ),
        PlanningStep(
            name='6._**basic_visualization:**_ensure_the_chart',
            description='6. **Basic Visualization:** Ensure the chart is clear, legible, and easy to understand.'
        )
    ],
    success_criteria=[
        'At least one valid canonical chart is produced, preferably among: Line Chart, Bar Chart.',
        'Generated charts are readable and consistent with the request.',
        'Final statements reference execution metrics or chart evidence.'
    ]
)

# NonCanonicalPlanningService

In [15]:
from src.services import NonCanonicalPlanningService

ncp = NonCanonicalPlanningService().invoke(runtime=runtime, query_understanding=qu)

In [16]:
rprint(ncp)

PlanningResult(
    mode=<ChartCaseType.NON_CANONICAL: 'non_canonical'>,
    steps=[
        PlanningStep(
            name='**1._initial_assessment_&_contextualization_(1',
            description='**1. Initial Assessment & Contextualization (1-2 hours)**'
        ),
        PlanningStep(
            name='**2._data_gathering_&_preparation_(2',
            description=' **2. Data Gathering & Preparation (2-4 hours)**'
        ),
        PlanningStep(
            name='**3._assumption_check_&_validation_(1',
            description=' **3. Assumption Check & Validation (1-2 hours)**'
        ),
        PlanningStep(
            name='**4._visualization_design_&_initial_draft',
            description=' **4. Visualization Design & Initial Draft (1-2 hours)**'
        ),
        PlanningStep(
            name='**5._preliminary_analysis_&_trend_identification',
            description=' **5. Preliminary Analysis & Trend Identification (4-8 hours)**'
        ),
        PlanningStep(
            name='**6._verification_&_refinement_(4_8',
            description=' **6. Verification & Refinement (4-8 hours)**'
        ),
        PlanningStep(
            name='**7._report_&_presentation_(1_2',
            description=' **7. Report & Presentation (1-2 hours)**'
        ),
        PlanningStep(
            name='**8._documentation_&_audit_(30_mins)**',
            description=' **8. Documentation & Audit (30 mins)**'
        )
    ],
    success_criteria=[
        'Clear, understandable visualization with minimal misleading trends.',
        'Robust evidence supporting the identified trends.',
        'Confirmation that the visualization accurately reflects the underlying data.',
        'Demonstrated adherence to established guidelines and reporting standards.',
        'Documentation of all assumptions, validation steps, and analysis performed.'
    ]
)

# DataProfilerService

In [17]:
from src.services import DataProfilerService

data_profile = DataProfilerService().invoke(runtime=runtime, data_path=data_path)

In [18]:
rprint(data_profile)

DataProfile(
    row_count=150,
    col_count=6,
    columns=[
        DataColumnProfile(name='Id', dtype='numeric', missing_ratio=0.0, unique_count=150),
        DataColumnProfile(name='SepalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=35),
        DataColumnProfile(name='SepalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=23),
        DataColumnProfile(name='PetalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=43),
        DataColumnProfile(name='PetalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=22),
        DataColumnProfile(name='Species', dtype='categorical', missing_ratio=0.0, unique_count=3)
    ],
    likely_numeric_columns=['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm'],
    likely_categorical_columns=['Species'],
    likely_time_columns=[],
    quality_notes=["Column 'Id' looks like an identifier."]
)

# DataPreparationService

In [19]:
from src.services import DataPreparationService

data_prep = DataPreparationService().invoke(runtime=runtime, data_path=data_path, data_profile=data_profile, run_id="1")

In [20]:
rprint(data_prep)

DataPreparationResult(
    output_path='D:/programming/projects/mas_rag/demo_data/tmp_folder/artifacts/1/cleaned_data.csv',
    operations=[],
    row_count=150,
    col_count=6
)

# VisRAG

In [21]:
from src.services import VisRAGService

recommendations = VisRAGService().invoke(runtime=runtime, data_profile=data_profile, query_understanding=qu)

In [22]:
rprint(recommendations)

VisRAGResult(
    recommendations=[
        VisRAGRecommendation(
            chart_family='line chart',
            rationale='Selected as a conservative fallback based on the request and available data shape.',
            priority=1,
            score=1.4,
            supporting_example_ids=[],
            supporting_corpora=[]
        ),
        VisRAGRecommendation(
            chart_family='bar chart',
            rationale='Selected as a conservative fallback based on the request and available data shape.',
            priority=2,
            score=1.2,
            supporting_example_ids=[],
            supporting_corpora=[]
        )
    ],
    rules=[
        'Use clear titles and axis labels.',
        'Avoid overcrowded visuals.',
        'Prefer readable defaults.',
        'Prefer chart families supported by both the data profile and retrieved reference examples.',
        'Validate whether a simpler canonical chart can communicate the same message.'
    ],
    caveats=['Time Period: Over time', 'Non-canonical case requires conservative interpretation.'],
    retrieved_examples=[],
    corpus_status=['No VisRAG corpus root configured; using heuristic-only recommendations.'],
    retrieval_strategy='hybrid_rule_retrieval'
)